# Real-Time Stream Processing Assignment — Run Notebook

Repository for Ontario Tech course **ENGR 5785G — Real-Time Data Analytics IoT**.

Student: Vitor Brandao Raposo

Student ID: 101011969

Date: 06/2026

Github link: https://github.com/MyLittleFoxxie/realtime_stream_processing_assignment

**Scenario B — Hospital Patient Monitoring (IoMT).** This notebook reproduces the
README steps end to end: set up the host environment, build the dataset, run the
Spark Structured Streaming job in Docker, feed the stream, and display the
`*** CLINICAL ALERT ***` output (the screenshot target for the assignment).

> Run the cells top to bottom. Spark runs **inside Docker**, so the Docker commands
> here drive the container; the Python steps (`prepare_data`, `producer`) run on the
> host via the project's `.venv`.

## Prerequisites

Two things must be on your `PATH`:

- **Docker Desktop** — runs the Spark job (the image bundles Java + PySpark).
- **[uv](https://docs.astral.sh/uv/)** — creates the host Python environment.

In [9]:
import subprocess, time, pathlib, shutil

PROJECT = pathlib.Path.cwd()
PY = str(PROJECT / ".venv" / "Scripts" / "python.exe")   # the .venv interpreter

def sh(cmd, env_extra=None):
    "Run a shell command, echo it, and stream back its output."
    print("$", cmd)
    import os
    env = {**os.environ, **(env_extra or {})}
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.stdout:
        print(r.stdout, end="")
    if r.returncode != 0 and r.stderr:
        print(r.stderr, end="")
    return r

def spark_logs():
    "Return the full Spark container log as a string (no -f / no following)."
    return subprocess.run("docker compose logs spark", shell=True,
                          text=True, capture_output=True).stdout

print("project:", PROJECT)
sh("docker --version")
sh("uv --version")

project: c:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment
$ docker --version
Docker version 29.2.0, build 0b9d198
$ uv --version
uv 0.11.15 (3cffe97c2 2026-05-18 x86_64-pc-windows-msvc)


CompletedProcess(args='uv --version', returncode=0, stdout='uv 0.11.15 (3cffe97c2 2026-05-18 x86_64-pc-windows-msvc)\n', stderr='')

## 1. First-time setup (run once)

Create the in-project virtual environment and install the host-side dependencies
(`pandas`, `openpyxl` — used only to read the Kaggle `.xlsx` during data prep).
Safe to re-run; `uv` is idempotent.

In [10]:
sh("uv venv")
# Point uv at the .venv we just created so the install lands there.
sh("uv pip install -r requirements.txt", env_extra={"VIRTUAL_ENV": str(PROJECT / ".venv")})

$ uv venv
Using CPython 3.11.15
Creating virtual environment at: .venv
error: Failed to create virtual environment
  Caused by: A virtual environment already exists at `.venv`. Use `--clear` to replace it
$ uv pip install -r requirements.txt


CompletedProcess(args='uv pip install -r requirements.txt', returncode=0, stdout='', stderr='\x1bChecked \x1b2 packages\x1b \x1bin 5ms\x1b\x1b\n')

## 2. Build the streamable dataset (one-time)

Turns the cross-sectional Kaggle `.xlsx` into `iomt_data/icu_readings.csv`, remapping
readings round-robin onto a pool of 30 ICU patients so patients recur over time.

> Requires the source file at `iomt_data/patients_data_with_alerts.xlsx` (gitignored,
> not submitted). If it's missing, download it first (see the README *Dataset* section).

In [11]:
sh(f'"{PY}" -m src.prepare_data')

$ "c:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment\.venv\Scripts\python.exe" -m src.prepare_data
[prepare] reading C:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment\iomt_data\patients_data_with_alerts.xlsx ...
[prepare] 50000 source readings loaded
[prepare] wrote 50000 readings -> C:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment\iomt_data\icu_readings.csv
[prepare] patient pool size: 30
[prepare] HR mean=104.5 bpm, >100 bpm: 54.6%


CompletedProcess(args='"c:\\Users\\Fox\\Desktop\\iot\\realtime_stream_processing_assignment\\.venv\\Scripts\\python.exe" -m src.prepare_data', returncode=0, stdout='[prepare] reading C:\\Users\\Fox\\Desktop\\iot\\realtime_stream_processing_assignment\\iomt_data\\patients_data_with_alerts.xlsx ...\n[prepare] 50000 source readings loaded\n[prepare] wrote 50000 readings -> C:\\Users\\Fox\\Desktop\\iot\\realtime_stream_processing_assignment\\iomt_data\\icu_readings.csv\n[prepare] patient pool size: 30\n[prepare] HR mean=104.5 bpm, >100 bpm: 54.6%\n', stderr='')

## 3. Clean runtime state (do this before every fresh run)

Spark records every input file it has already ingested in its checkpoint, and the
producer reuses the same `tick_*.json` filenames each run. So without wiping the
runtime directories, Spark resumes from the checkpoint, skips the re-emitted files,
and prints **no new windows or alerts**. This cell stops the container and removes
the gitignored runtime dirs.

In [12]:
sh("docker compose down")
for d in ["checkpoints", "stream_input", "stream_window_out", "alerts"]:
    shutil.rmtree(PROJECT / d, ignore_errors=True)
    print("removed", d)

$ docker compose down
removed checkpoints
removed stream_input
removed stream_window_out
removed alerts


## 4. Start the Spark job and wait until it is watching

`docker compose up -d` launches the streaming query detached. We then poll the logs
until the `[spark] watching ...` line appears — the producer must only start **after**
Spark is watching the directory.

In [13]:
sh("docker compose up -d")

ready = False
for _ in range(40):                       # up to ~80s
    time.sleep(2)
    logs = spark_logs()
    if "[spark] watching" in logs:
        ready = True
        break

print("Spark ready:", ready)
print(next((l for l in logs.splitlines() if "watching" in l), "(watching line not found yet)"))

$ docker compose up -d
Spark ready: True
spark_icu  | [spark] watching /app/stream_input | window=2 minutes watermark=4 minutes threshold=100 bpm


## 5. Feed the stream

The producer writes one JSON file per *tick* (one reading per patient) into
`stream_input/`, ~1 tick/second, advancing event-time 30s per tick. It ends with a
far-future, below-threshold **flush tick** that advances the watermark so Spark
finalizes every real window. This cell blocks for ~1 minute while it runs.

In [14]:
sh(f'"{PY}" -m src.producer')

$ "c:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment\.venv\Scripts\python.exe" -m src.producer
[producer] 30 patients, emitting 60 ticks into C:\Users\Fox\Desktop\iot\realtime_stream_processing_assignment\stream_input
[producer] event-time step 30s (4 ticks per 2 minutes window)
[producer] tick   0  event_time=2026-06-14T00:00:00+00:00  (30 readings)
[producer] tick   1  event_time=2026-06-14T00:00:30+00:00  (30 readings)
[producer] tick   2  event_time=2026-06-14T00:01:00+00:00  (30 readings)
[producer] tick   3  event_time=2026-06-14T00:01:30+00:00  (30 readings)
[producer] tick   4  event_time=2026-06-14T00:02:00+00:00  (30 readings)
[producer] tick   5  event_time=2026-06-14T00:02:30+00:00  (30 readings)
[producer] tick   6  event_time=2026-06-14T00:03:00+00:00  (30 readings)
[producer] tick   7  event_time=2026-06-14T00:03:30+00:00  (30 readings)
[producer] tick   8  event_time=2026-06-14T00:04:00+00:00  (30 readings)
[producer] tick   9  event_time=2026-06-14T00:04:3

CompletedProcess(args='"c:\\Users\\Fox\\Desktop\\iot\\realtime_stream_processing_assignment\\.venv\\Scripts\\python.exe" -m src.producer', returncode=0, stdout='[producer] 30 patients, emitting 60 ticks into C:\\Users\\Fox\\Desktop\\iot\\realtime_stream_processing_assignment\\stream_input\n[producer] event-time step 30s (4 ticks per 2 minutes window)\n[producer] tick   0  event_time=2026-06-14T00:00:00+00:00  (30 readings)\n[producer] tick   1  event_time=2026-06-14T00:00:30+00:00  (30 readings)\n[producer] tick   2  event_time=2026-06-14T00:01:00+00:00  (30 readings)\n[producer] tick   3  event_time=2026-06-14T00:01:30+00:00  (30 readings)\n[producer] tick   4  event_time=2026-06-14T00:02:00+00:00  (30 readings)\n[producer] tick   5  event_time=2026-06-14T00:02:30+00:00  (30 readings)\n[producer] tick   6  event_time=2026-06-14T00:03:00+00:00  (30 readings)\n[producer] tick   7  event_time=2026-06-14T00:03:30+00:00  (30 readings)\n[producer] tick   8  event_time=2026-06-14T00:04:00+00

## Written explanation

**Why a tumbling window.** Scenario B needs the average heart rate _per patient per
window_ and an alert on _two consecutive windows_. A **tumbling** (fixed,
non-overlapping) 2-minute window gives each patient exactly **one** average per
interval, so "two consecutive windows" maps cleanly to two adjacent, non-overlapping
intervals (`start2 == start1 + 2 min`). A sliding window would reuse readings across
overlapping windows, double-counting samples and inflating false alerts for what is meant to be a _sustained_ signal.

**Where the pipeline requires state.** Two places:

1. **Windowed-aggregation state** — a running
   `avg(heart_rate)` per `(patient_id, 2-min window)`, held until the **watermark**
   (`event_time - 4 min`) passes the window's end, then emitted once and evicted. The
   watermark bounds this state so it can't grow without limit.
2. **Consecutive-breach state** — a per-patient memory of the
   most recent breaching window's start, which spans windows and micro-batches and so
   isn't expressible as a single aggregation. The `foreachBatch` handler keeps a
   `patient_id -> last_breaching_window_start` dict and fires the alert when the
   current breaching window is adjacent to the stored one.

## 6. Streaming output (screenshot target)

Spark runs as a series of **micro-batches**: on each 5-second trigger it ingests new
files and, in *append* mode, emits a 2-minute window only once the watermark passes
the window's end. We wait a few seconds for the flush to finalize the last windows,
then pull the alert lines out of the container log.

In [15]:
time.sleep(15)                            # let the final flush batch finalize
logs = spark_logs()
lines = logs.splitlines()

batch_headers = [l for l in lines if "===== batch" in l]
alerts = [l for l in lines if "CLINICAL ALERT" in l]

print(f"micro-batches that finalized windows: {len(batch_headers)}")
print(f"clinical alerts fired: {len(alerts)}")
print("\n--- batch headers ---")
for l in batch_headers:
    print(l)
print("\n--- clinical alerts (first 15) ---")
for l in alerts[:15]:
    print(l)

micro-batches that finalized windows: 10
clinical alerts fired: 167

--- batch headers ---
spark_icu  | ===== batch 3: 30 finalized window(s) =====
spark_icu  | ===== batch 4: 30 finalized window(s) =====
spark_icu  | ===== batch 5: 30 finalized window(s) =====
spark_icu  | ===== batch 6: 30 finalized window(s) =====
spark_icu  | ===== batch 7: 60 finalized window(s) =====
spark_icu  | ===== batch 8: 30 finalized window(s) =====
spark_icu  | ===== batch 9: 60 finalized window(s) =====
spark_icu  | ===== batch 10: 30 finalized window(s) =====
spark_icu  | ===== batch 12: 60 finalized window(s) =====
spark_icu  | ===== batch 13: 90 finalized window(s) =====

--- clinical alerts (first 15) ---
spark_icu  |   *** CLINICAL ALERT *** patient 2: avg HR > 100 bpm in two consecutive windows (00:00-00:02), latest avg 106.5 bpm
spark_icu  |   *** CLINICAL ALERT *** patient 4: avg HR > 100 bpm in two consecutive windows (00:00-00:02), latest avg 114.8 bpm
spark_icu  |   *** CLINICAL ALERT *** pati

### Inspect the alert CSV the job wrote

In [16]:
import pandas as pd
alerts_csv = PROJECT / "alerts" / "clinical_alerts.csv"
pd.read_csv(alerts_csv) if alerts_csv.exists() else "no alerts file yet"

,patient_id,prev_window_start,curr_window_start,avg_hr
0,2,2026-06-14T00:00:00,2026-06-14T00:02:00,106.50
1,4,2026-06-14T00:00:00,2026-06-14T00:02:00,114.75
2,8,2026-06-14T00:00:00,2026-06-14T00:02:00,106.00
3,10,2026-06-14T00:00:00,2026-06-14T00:02:00,113.00
4,11,2026-06-14T00:00:00,2026-06-14T00:02:00,102.75
...,...,...,...,...
162,26,2026-06-14T00:26:00,2026-06-14T00:28:00,102.25
163,27,2026-06-14T00:26:00,2026-06-14T00:28:00,101.25
164,28,2026-06-14T00:22:00,2026-06-14T00:24:00,107.00
165,29,2026-06-14T00:24:00,2026-06-14T00:26:00,108.00


## 7. Tear down

Stop and remove the Spark container. (Re-running the whole notebook? Go back to
**step 3** to clean runtime state first.)

In [ ]:
sh("docker compose down")